# 第4段階　Fold 3の誤差と新特徴量の追加効果
E1からE4を実行します。第3段階に保存したモデルを読み込み、学習内・将来区間のスコアを比較します。窓構成・入力署名・保存予測が一致することを確認します。評価窓の確率を反転したり、外部データで閾値を調整する処理はありません。

改善案は既存20項目を使うL2ロジスティック回帰1種類です。各FoldでモデルAの学習内AUCが0.75以上のとき、標本内で分離できている関係をより単純なモデルでも捉えられるかを試します。内部の先頭2/3→末尾1/3での参考AUC、補完・標準化は学習範囲内。正則化係数C=1・閾値0.5は固定。外部は第3段階と同じFold 3の固定モデルを評価します。

In [ ]:
# E1　Drive接続・固定コードの取得・合成データの検査
from google.colab import drive
from pathlib import Path
import sys, os, subprocess, tempfile, urllib.request, importlib.util
drive.mount('/content/drive')
PIN = '3cf6b55544e11f06de9df8c6d445750b10ec0d67'
src = Path(tempfile.mkdtemp(prefix='mahjong_model_'))
files = ['mahjong_research/'+f for f in ('__init__.py','analysis.py','inputs.py','tenhou.py','legacy.py')]
files += ['smoothing/study.py','smoothing/presentation.py','smoothing/test_smoothing.py',
          'stability/stability.py','choices/choices.py']
files += ['model_comparison/'+f for f in ('windows.py','compare.py','outputs.py','test_comparison.py')]
files += ['error_analysis/'+f for f in ('investigate.py','diagnostics.py','findings.py','test_error_analysis.py')]
for rel in files:
    target=src/rel
    target.parent.mkdir(exist_ok=True)
    url=f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{PIN}/research_v2/{rel}'
    with urllib.request.urlopen(url,timeout=60) as r: target.write_bytes(r.read())
if importlib.util.find_spec('lightgbm') is None:
    subprocess.run([sys.executable,'-m','pip','install','lightgbm==4.6.0'],check=True)
paths=[str(src/p) for p in ('error_analysis','model_comparison','stability','smoothing','choices','.')]
env=dict(os.environ,PYTHONPATH=os.pathsep.join(paths),MPLBACKEND='Agg')
subprocess.run([sys.executable,'-m','unittest','-q',
                'test_comparison.ComparisonTests.test_saved_phase1_and_phase2_assembly',
                'test_comparison.ComparisonTests.test_selection_is_past_only',
                'test_error_analysis.ErrorAnalysisTests.test_same_marginal_reversed_labels',
                'test_error_analysis.ErrorAnalysisTests.test_train_only_imputation_and_scaling'],cwd=src,env=env,check=True)

In [ ]:
# E2　Driveの入力フォルダを指定。第2段階は第3段階の入力ハッシュで照合
import hashlib, json
ROOT=Path('/content/drive/MyDrive/卒論')
RUN=ROOT/'research_v2_runs/20261105T071722Z_5da6c9ad'
PHASE1=ROOT/'research_v2_stability/20261105T092459Z_113b116b'
PHASE3=ROOT/'research_v2_model_comparison/20261117T160007Z_113bc0ca'
PHASE2=None  # 手動指定時はQ5完了フォルダのフルパス
ULTRA_CACHE=None  # C8キャッシュを移動した場合のみ指定
OUTPUT=ROOT/'research_v2_error_analysis'
if PHASE2 is None:
    signature=json.loads((PHASE3/'protocol.json').read_text())['inputs']['window_values.csv']
    matches=[p.parent for p in sorted((ROOT/'research_v2_choices').glob('*/window_values.csv'))
             if hashlib.sha256(p.read_bytes()).hexdigest()==signature]
    if not matches: raise FileNotFoundError('第3段階で使用したQ5出力をPHASE2に指定してください')
    PHASE2=matches[0]
print('Q5入力', PHASE2, '保存済みモデル',PHASE3/'models')

In [ ]:
# E3　保存済み予測照合・原因分析・条件に該当するFoldでの限定比較
args=[str(p) for p in (RUN,PHASE1,PHASE2,PHASE3,OUTPUT)]+[str(ULTRA_CACHE) if ULTRA_CACHE else None]
code="""import sys,json
from pathlib import Path
from investigate import run
out=run(*json.loads(sys.argv[1]))
Path(sys.argv[2]).write_text(str(out),encoding='utf8')
"""
result_path=src/'result_path.txt'
subprocess.run([sys.executable,'-u','-c',code,json.dumps(args),str(result_path)],cwd=src,env=env,check=True)
RESULT=Path(result_path.read_text(encoding='utf8'))
print('保存先',RESULT)

In [ ]:
# E4　Fold別の根拠と改善前後の性能を表示
import pandas as pd
from IPython.display import display,Markdown,Image
display(Markdown((RESULT/'report_ja.md').read_text(encoding='utf8')))
display(pd.read_csv(RESULT/'improvement_metrics.csv'))
display(pd.read_csv(RESULT/'improvement_comparison.csv'))
for name in ['feature_distributions.png','score_distributions.png','improvement.png']:
    display(Image(filename=str(RESULT/name)))